# ViFinQA — finish a run that a session cut short

This notebook only resumes. It needs no dense index, no hybrid retrieval and no reranking,
because the run it continues already produced the retrieval those stages exist to build, and
the generation checkpoint is keyed to that exact file.

Attach four inputs:

1. `vifinqa` — the corpus, for the evidence CSVs.
2. `vifinqa-artifacts` — the frozen table manifest.
3. the checkpoint dataset built from the earlier run's output, holding
   `artifacts/retrieval_reranked.jsonl` (with its `.metadata.json`) and the `rows/`
   directories under `artifacts/generation_<model_profile>_shards/`.

The evidence CSVs under each shard's `data/` do **not** need to travel. They are a
deterministic function of the corpus and the manifest, they are what makes the output
too large to download, and this notebook rebuilds them.

Then run every cell in order. Nothing here is optional and nothing needs a flag flipped: a
run reaches this notebook only after it was already declared final.


In [ ]:
# Pins carried over from the run being resumed. The checkpoint is keyed to them, so a
# resumed session that changes any of these starts again from nothing.
import os

os.environ["VIFINQA_THINKING_MODE"] = "disabled"
os.environ["VIFINQA_TP"] = "1"
os.environ["VIFINQA_DP"] = "2"
os.environ["VIFINQA_SHARDS_PER_REPLICA"] = "4"
# Answer at most this many questions counting from the first, then stop cleanly so the
# version completes and its output can be attached to the session after it. Empty means
# finish the run and package the submission.
os.environ["VIFINQA_QUESTION_LIMIT"] = ""
print("resume shards:", 2 * 4, "| model is read from checkpoint metadata")


In [ ]:
import base64
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import requests
import torch

def iter_input_paths(
    relative: str, root: Path = Path("/kaggle/input"), max_depth: int = 12
) -> list[Path]:
    """Return every existing `<directory>/relative` under `root`, following symlinked mounts."""
    matches: list[Path] = []
    visited: set[str] = set()
    for parent, directories, _ in os.walk(root, followlinks=True):
        real = os.path.realpath(parent)
        if real in visited:
            directories.clear()
            continue
        visited.add(real)
        if len(Path(parent).parts) - len(root.parts) >= max_depth:
            directories.clear()
        candidate = Path(parent) / relative
        if candidate.exists():
            matches.append(candidate)
    return sorted(matches, key=str)


def describe_inputs(root: Path = Path("/kaggle/input"), max_depth: int = 5) -> str:
    """Return a compact inventory of mounted inputs so failures name what is actually attached.

    Kaggle spends three levels on `datasets/<owner>/<slug>` before any content, so the
    default depth has to reach past the mount point itself.
    """
    if not root.is_dir():
        return f"{root} does not exist"
    lines: list[str] = []
    visited: set[str] = set()
    for parent, directories, filenames in os.walk(root, followlinks=True):
        real = os.path.realpath(parent)
        if real in visited:
            directories.clear()
            continue
        visited.add(real)
        depth = len(Path(parent).parts) - len(root.parts)
        entries = sorted(filenames)[:4]
        if len(filenames) > 4:
            entries.append(f"+{len(filenames) - 4} more files")
        if depth >= max_depth:
            if directories:
                entries.append(f"+{len(directories)} more directories")
            directories.clear()
        directories.sort()
        lines.append(f"{'  ' * depth}{Path(parent).name or root}/ {entries}")
        if len(lines) >= 80:
            lines.append("... truncated")
            break
    return "\n".join(lines)


def make_writable(root: Path) -> None:
    """Kaggle mounts inputs read-only and copytree preserves that.

    A resumed run has to append rows to the checkpoint it inherited, so the copy has to be
    writable even though the original never was.
    """
    for path in [root, *root.rglob("*")]:
        path.chmod(path.stat().st_mode | (0o700 if path.is_dir() else 0o600))


try:
    from kaggle_secrets import UserSecretsClient

    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    del hf_token

print(
    "torch", torch.__version__, "cuda", torch.cuda.is_available(), "gpus", torch.cuda.device_count()
)
assert torch.cuda.is_available(), "Enable a GPU accelerator before continuing."
DP = int(os.environ["VIFINQA_DP"])
TP = int(os.environ["VIFINQA_TP"])
SHARDS = DP * int(os.environ["VIFINQA_SHARDS_PER_REPLICA"])
assert torch.cuda.device_count() >= DP, "Select T4 x2."
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    capability = torch.cuda.get_device_capability(index)
    print(index, properties.name, round(properties.total_memory / 2**30, 1), "GiB", "sm", capability)
    assert capability >= (7, 5), f"GPU {index} is older than sm_75."
THINKING_MODE = os.environ["VIFINQA_THINKING_MODE"]


In [ ]:
INPUT_INVENTORY = describe_inputs()
print("Kaggle inputs:\n" + INPUT_INVENTORY)

data_candidates = sorted(
    {
        questions.parent.parent
        for questions in iter_input_paths("questions/questions.jsonl")
        if (questions.parent.parent / "code_stock.csv").is_file()
        and (questions.parent.parent / "financial_statements").is_dir()
    },
    key=str,
)
assert data_candidates, f"Attach the `vifinqa` corpus.\n{INPUT_INVENTORY}"
DATA_ROOT = data_candidates[0]

manifests = sorted(
    {
        manifest
        for manifest in iter_input_paths("processed/table_manifest.jsonl")
        if manifest.with_suffix(".parquet").is_file()
    },
    key=str,
)
assert manifests, f"Attach the `vifinqa-artifacts` manifest.\n{INPUT_INVENTORY}"
MANIFEST = manifests[0]


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


# The evidence a resumed run writes has to come from the same tables the earlier one read.
assert (
    sha256(MANIFEST.with_suffix(".parquet"))
    == "060bd26eff14d30ce70b3ba7b00af509be6100b58ddb5a0fe970afa0ef69e29d"
), "The manifest differs from the one the checkpoint was built against."

# Locate exactly one supported model checkpoint, then derive model/revision from its
# metadata instead of trusting a manually copied notebook setting.
RESUME_PROFILES = {
    "qwen3_8b_awq": (
        "Qwen/Qwen3-8B-AWQ",
        "4da05a8edb55c6046cce958586c33b61da07bb79",
        4,
    ),
    "qwen3_14b_awq": (
        "Qwen/Qwen3-14B-AWQ",
        "31c69efc29464b6bb0aee1398b5a7b50a99340c3",
        2,
    ),
}
checkpoint_candidates = [
    (profile, path)
    for profile in RESUME_PROFILES
    for path in iter_input_paths(f"generation_{profile}_shards/shard_0/run_metadata.json")
]
# Two finished sessions can be attached at once for good reason -- the earlier one may hold
# the hybrid ranking or diagnostics the later one no longer writes -- so refusing outright
# makes the user detach an input they need. Name the one you mean instead. The guard still
# refuses when nothing is named and more than one is present, because resuming the wrong run
# silently is worse than stopping.
WANTED_CHECKPOINT = os.environ.get("VIFINQA_CHECKPOINT_SOURCE", "").strip()
if WANTED_CHECKPOINT:
    checkpoint_candidates = [
        candidate for candidate in checkpoint_candidates if WANTED_CHECKPOINT in str(candidate[1])
    ]
assert len(checkpoint_candidates) == 1, (
    f"Attach exactly one supported generation checkpoint, or name one with VIFINQA_CHECKPOINT_SOURCE (a substring of its path). Found {checkpoint_candidates}.\n"
    f"{INPUT_INVENTORY}"
)
MODEL_PROFILE, prior_metadata_path = checkpoint_candidates[0]
MODEL, MODEL_REVISION, DEFAULT_MAX_NUM_SEQS = RESUME_PROFILES[MODEL_PROFILE]
prior_metadata = json.loads(prior_metadata_path.read_text(encoding="utf-8"))
assert prior_metadata.get("model") == MODEL
assert prior_metadata.get("model_revision") == MODEL_REVISION
TABLE_UNIT_SOURCE = str(prior_metadata.get("table_unit_source", "latest"))
assert TABLE_UNIT_SOURCE in {"manifest", "latest"}
# Read the breadth from the run being resumed rather than from a constant here. Both
# values feed the fingerprint, and a fingerprint mismatch discards every question the
# earlier session completed, so the two notebooks must not hold the number separately.
CANDIDATE_TABLES = str(prior_metadata["candidate_tables"])
# Both feed the fingerprint too, and both were being left to the script's defaults here.
# The first notebook passed 4096 while this one defaulted to 6144, so a resume of its
# output was refused for a setting nobody had chosen to change. Read them from what that
# run recorded, the same way the candidate count already is.
# Same story: the scope router decides which candidates the prompt ever showed, so a
# resume that assumed a different policy would answer the second half of the run from a
# different set of tables than the first.
SCOPE_ROUTER = str(prior_metadata.get("scope_router", "both"))
# Read back, never re-declared. These three are in the fingerprint, so a session that finishes
# a run has to reproduce the values that run was started with or the checkpoint is refused --
# after the model is loaded and the retrieval rebuilt, which is the expensive place to find out.
ROOT_GRAMMAR = str(prior_metadata.get("root_grammar", "off"))
WORKED_EXAMPLE = bool(prior_metadata.get("worked_example", False))
REPAIR_YEAR_ANSWER = bool(prior_metadata.get("repair_year_answer", False))
MAX_TOKENS = str(prior_metadata["max_tokens"])
CONTEXT_LIMIT = str(prior_metadata["context_limit"])
os.environ["VIFINQA_MODEL"] = MODEL
os.environ["VIFINQA_MODEL_REVISION"] = MODEL_REVISION
os.environ.setdefault("VIFINQA_MAX_NUM_SEQS", str(DEFAULT_MAX_NUM_SEQS))

# Reuse whichever retrieval the earlier run actually used. Reproducing it would be cheaper
# to write and far more dangerous: one byte of drift discards every completed question.
retrieval_candidates = [
    path
    for path in [
        *iter_input_paths("retrieval_reranked.jsonl"),
        *iter_input_paths("retrieval_hybrid.jsonl"),
        *iter_input_paths("retrieval_bm25.jsonl"),
    ]
    if sha256(path) == prior_metadata.get("retrieval_sha256")
]
assert len(retrieval_candidates) == 1, (
    "Checkpoint dataset must carry exactly the retrieval whose SHA is in run_metadata."
)
finished = retrieval_candidates
ARTIFACTS = Path("/kaggle/working/artifacts")
ARTIFACTS.mkdir(parents=True, exist_ok=True)
RETRIEVAL = ARTIFACTS / "retrieval_reranked.jsonl"
for suffix in ("", ".metadata.json"):
    source = finished[0].with_name(finished[0].name + suffix)
    assert source.is_file(), f"Missing retrieval provenance file: {source}"
    shutil.copy2(
        source,
        RETRIEVAL.with_name(RETRIEVAL.name + suffix),
    )
    RETRIEVAL.with_name(RETRIEVAL.name + suffix).chmod(0o644)
retrieval_rows = sum(1 for line in RETRIEVAL.read_text(encoding="utf-8").splitlines() if line)
assert retrieval_rows == 1012, f"Retrieval has {retrieval_rows} rows, expected 1012."
print("reusing retrieval:", finished[0], sha256(RETRIEVAL))

GENERATION_NAME = f"generation_{MODEL_PROFILE}"
GEN = ARTIFACTS / GENERATION_NAME
GEN_SHARDS = ARTIFACTS / f"{GENERATION_NAME}_shards"
prior = [prior_metadata_path]
PROJECT_SHA = prior_metadata["project_revision"]
assert len(PROJECT_SHA) == 40, f"Checkpoint records no project revision: {PROJECT_SHA}"
if not GEN_SHARDS.exists():
    shutil.copytree(prior[0].parents[1], GEN_SHARDS)
    make_writable(GEN_SHARDS)


def completed_rows() -> int:
    """Count answers the way the merge counts them.

    Globbing the checkpoint directory reported zero on a run whose shards had all just
    reported completion, which turned a finished run into a failed session. The consolidated
    files are what the merge reads, so let them be what decides whether a run is done.
    """
    total = 0
    for shard in sorted(GEN_SHARDS.glob("shard_*")):
        seen: set[int] = set()
        for name in ("predictions.jsonl", "errors.jsonl"):
            path = shard / name
            if not path.is_file():
                continue
            for line in path.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    seen.add(int(json.loads(line)["id"]))
        if not seen:
            seen = {int(item.stem) for item in (shard / "completed/rows").glob("*.json")}
        total += len(seen)
    return total


shard_count = len(list(GEN_SHARDS.glob("shard_*")))
assert shard_count == SHARDS, (
    f"The checkpoint was written by {shard_count} shards and this session would use "
    f"{SHARDS}. Set VIFINQA_SHARDS_PER_REPLICA so they match, or the run starts over."
)
print(f"resuming with {completed_rows()}/1012 questions already answered")
print("checkpoint project revision:", PROJECT_SHA)


In [ ]:
# The checkpoint names the commit that wrote it, so check out exactly that one.
GIT_URL = "https://github.com/ThanhDatVN/AI-Financial-Data-Assistant.git"
PROJECT = Path("/kaggle/working/AI-Financial-Data-Assistant")
if PROJECT.exists() and not (PROJECT / "pyproject.toml").exists():
    shutil.rmtree(PROJECT)
if not PROJECT.exists():
    result = subprocess.run(
        ["git", "clone", GIT_URL, str(PROJECT)], capture_output=True, text=True
    )
    if result.returncode:
        try:
            from kaggle_secrets import UserSecretsClient

            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception as exc:
            raise RuntimeError((result.stderr or "git clone failed").strip()) from exc
        auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        git_env = os.environ.copy()
        git_env.update(
            {
                "GIT_CONFIG_COUNT": "1",
                "GIT_CONFIG_KEY_0": "http.extraHeader",
                "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}",
            }
        )
        subprocess.run(["git", "clone", GIT_URL, str(PROJECT)], env=git_env, check=True)
        del token, auth, git_env
# A full clone already holds every commit on main. Fetching a bare SHA depends on a
# server-side setting, so let it fail quietly rather than take the session with it.
subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin", PROJECT_SHA], check=False)
subprocess.run(["git", "-C", str(PROJECT), "checkout", PROJECT_SHA], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements-gpu.txt")],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT)], check=True)
os.chdir(PROJECT)
checked_out = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert checked_out == PROJECT_SHA, f"Checked out {checked_out}, checkpoint wants {PROJECT_SHA}"
print("project revision:", checked_out)
Path("/kaggle/working/runtime_environment.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8"
)


In [ ]:
# Replicate the model once per T4; full generation sends two independent shards.
#
# Qwen3-14B-AWQ is the tighter fit. Weights land near 8.6 GiB of the 13.8 GiB that
# --gpu-memory-utilization 0.90 leaves on a T4, and its KV cache costs 160 KiB per token
# (40 layers x 8 KV heads x 128 head_dim x 2 x fp16). That leaves room for roughly 34k
# cached tokens, so --max-model-len 16384 fits about two sequences at once rather than the
# four Qwen3-8B-AWQ afforded. MAX_NUM_SEQS defaults down accordingly; raise it only after a
# smoke run shows headroom, because over-subscribing makes vLLM preempt and recompute.
TP = int(os.environ.get("VIFINQA_TP", "1"))
MAX_NUM_SEQS = int(os.environ.get("VIFINQA_MAX_NUM_SEQS", str(DEFAULT_MAX_NUM_SEQS)))
DP = int(os.environ.get("VIFINQA_DP", "2"))
assert TP in {1, 2}, "Tensor parallelism above 2 has no second pair of T4s to use."
assert 1 <= DP <= torch.cuda.device_count(), "VIFINQA_DP must not exceed the GPU count."
# Two client shards per replica keep a batch forming while one shard validates and
# executes the program it just received.
SHARDS = DP * int(os.environ.get("VIFINQA_SHARDS_PER_REPLICA", "2"))
# The generator sizes each question's token budget against this, so the server and
# the client must read the same number rather than two copies of it.
MAX_MODEL_LEN = 16384
VLLM_BASE = "http://127.0.0.1:8000"
VLLM_CONFIG = Path("/kaggle/working/vllm_server_config.json")
expected_server_config = {
    "model": MODEL,
    "model_revision": MODEL_REVISION,
    "tensor_parallel_size": TP,
    "data_parallel_size": DP,
    "max_num_seqs": MAX_NUM_SEQS,
    "max_model_len": MAX_MODEL_LEN,
    "quantization": "awq_marlin",
}


def cuda_driver_linker_environment() -> dict[str, str]:
    # Kaggle exposes libcuda.so.1 at runtime but its CUDA image can omit the
    # unversioned libcuda.so linker name needed by FlashInfer JIT.
    candidates = []
    ldconfig = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True, check=False)
    for line in ldconfig.stdout.splitlines():
        if "libcuda.so.1" in line and "=>" in line:
            candidates.append(Path(line.rsplit("=>", 1)[1].strip()))
    candidates.extend(
        [
            Path("/usr/lib/x86_64-linux-gnu/libcuda.so.1"),
            Path("/usr/local/nvidia/lib64/libcuda.so.1"),
            Path("/usr/lib64/libcuda.so.1"),
        ]
    )
    driver_library = next((path for path in candidates if path.is_file()), None)
    if driver_library is None:
        raise RuntimeError("CUDA driver is active but libcuda.so.1 was not found.")

    linker_dir = Path("/kaggle/working/cuda-driver-link")
    linker_dir.mkdir(parents=True, exist_ok=True)
    linker_name = linker_dir / "libcuda.so"
    if linker_name.is_symlink() or linker_name.exists():
        linker_name.unlink()
    linker_name.symlink_to(driver_library.resolve())

    environment = os.environ.copy()
    for variable in ("LIBRARY_PATH", "LD_LIBRARY_PATH"):
        existing = [item for item in environment.get(variable, "").split(os.pathsep) if item]
        environment[variable] = os.pathsep.join(
            [str(linker_dir), *[item for item in existing if item != str(linker_dir)]]
        )

    probe_path = linker_dir / "cuda_link_probe"
    probe = subprocess.run(
        ["c++", "-x", "c++", "-", f"-L{linker_dir}", "-lcuda", "-o", str(probe_path)],
        input="int main() { return 0; }\n",
        capture_output=True,
        text=True,
        env=environment,
    )
    probe_path.unlink(missing_ok=True)
    if probe.returncode:
        raise RuntimeError(f"CUDA driver linker probe failed:\n{probe.stderr}")
    print("verified CUDA driver linker:", linker_name, "->", driver_library)
    return environment


def served_model_ids() -> set[str]:
    try:
        health = requests.get(f"{VLLM_BASE}/health", timeout=2)
        if not health.ok:
            return set()
        response = requests.get(f"{VLLM_BASE}/v1/models", timeout=5)
        response.raise_for_status()
        return {item["id"] for item in response.json().get("data", [])}
    except (requests.RequestException, KeyError, TypeError, ValueError):
        return set()


existing_models = served_model_ids()
if existing_models:
    assert (
        MODEL in existing_models
    ), f"Port 8000 already serves {sorted(existing_models)}, not {MODEL}. Restart the session."
    assert (
        VLLM_CONFIG.exists()
    ), "A pre-existing vLLM server has unknown TP/DP; restart the session."
    actual_server_config = json.loads(VLLM_CONFIG.read_text(encoding="utf-8"))
    assert (
        actual_server_config == expected_server_config
    ), f"Existing vLLM config {actual_server_config} != {expected_server_config}; restart."
    print("reusing healthy vLLM server:", sorted(existing_models))
else:
    server_environment = cuda_driver_linker_environment()
    server_log = open("/kaggle/working/vllm.log", "a", encoding="utf-8")  # noqa: SIM115
    serve_cmd = [
        "vllm",
        "serve",
        MODEL,
        "--served-model-name",
        MODEL,
        "--host",
        "127.0.0.1",
        "--port",
        "8000",
        "--tensor-parallel-size",
        str(TP),
        "--data-parallel-size",
        str(DP),
        "--api-server-count",
        "1",
        "--dtype",
        "half",
        "--quantization",
        # vLLM prints "Detected that the model can run with awq_marlin, however you
        # specified quantization=awq explicitly, so forcing awq. Use quantization=awq_marlin
        # for faster inference" -- the fast kernel was available on this T4 all along and the
        # explicit flag was turning it off. Decode ran at 3.33 tok/s, which is what made a
        # 2048-token budget unreachable inside a 360 s request timeout.
        "awq_marlin",
        "--max-model-len",
        "16384",
        "--max-num-seqs",
        str(MAX_NUM_SEQS),
        "--gpu-memory-utilization",
        "0.90",
        "--generation-config",
        "vllm",
        "--default-chat-template-kwargs",
        '{"enable_thinking": false}',
        "--seed",
        "20260802",
    ]
    if MODEL_REVISION:
        serve_cmd += ["--revision", MODEL_REVISION]
    server = subprocess.Popen(
        serve_cmd, stdout=server_log, stderr=subprocess.STDOUT, env=server_environment
    )

    for _ in range(120):
        if MODEL in served_model_ids():
            break
        if server.poll() is not None:
            log_tail = Path("/kaggle/working/vllm.log").read_text(
                encoding="utf-8", errors="replace"
            )[-50_000:]
            raise RuntimeError(log_tail)
        time.sleep(5)
    else:
        raise TimeoutError("vLLM did not become healthy; inspect /kaggle/working/vllm.log")
    VLLM_CONFIG.write_text(json.dumps(expected_server_config, indent=2) + "\n", encoding="utf-8")
    print("vLLM ready:", MODEL, MODEL_REVISION, "tp/dp=", TP, DP)

In [ ]:
# The answers a run produced cannot be recreated; the tables they cite can. Carrying the
# evidence CSVs between sessions costs gigabytes, so a checkpoint dataset only has to move
# the rows, and this rebuilds what those rows point at.
subprocess.run(
    [
        sys.executable,
        "scripts/52_restore_evidence_csv.py",
        *[str(GEN_SHARDS / f"shard_{index}") for index in range(SHARDS)],
        "--manifest",
        str(MANIFEST.with_suffix(".parquet")),
        "--data-root",
        str(DATA_ROOT),
    ],
    check=True,
)


In [ ]:
# Resume the shards. Each writes one row at a time, so a session that ends early costs
# only the questions in flight.
common_cmd = [
    sys.executable,
    "scripts/50_generate_programs.py",
    "--retrieval",
    str(RETRIEVAL),
    "--manifest",
    str(MANIFEST.with_suffix(".parquet")),
    "--data-root",
    str(DATA_ROOT),
    "--model",
    MODEL,
    "--model-revision",
    MODEL_REVISION,
    "--thinking-mode",
    THINKING_MODE,
    "--table-unit-source",
    TABLE_UNIT_SOURCE,
    # Every field below feeds the run fingerprint, and a resume whose fingerprint
    # differs is refused outright. Omitting this one fell back to the script default
    # of ten, so a run started at twenty could never be finished here.
    "--candidate-tables",
    CANDIDATE_TABLES,
    "--scope-router",
    SCOPE_ROUTER,
    "--root-grammar",
    ROOT_GRAMMAR,
    "--max-tokens",
    MAX_TOKENS,
    "--context-limit",
    CONTEXT_LIMIT,
    "--max-attempts",
    "3",
    "--project-revision",
    PROJECT_SHA,
    "--final-run",
]
if WORKED_EXAMPLE:
    common_cmd += ["--worked-example"]
if REPAIR_YEAR_ANSWER:
    common_cmd += ["--repair-year-answer"]
QUESTION_LIMIT = os.environ.get("VIFINQA_QUESTION_LIMIT", "").strip()
if QUESTION_LIMIT:
    common_cmd += ["--limit", QUESTION_LIMIT]
shard_dirs = [GEN_SHARDS / f"shard_{index}" for index in range(SHARDS)]
workers = []
for shard_index, shard_dir in enumerate(shard_dirs):
    workers.append(
        (
            shard_index,
            subprocess.Popen(
                common_cmd
                + [
                    "--output",
                    str(shard_dir),
                    "--shard-count",
                    str(SHARDS),
                    "--shard-index",
                    str(shard_index),
                ]
            ),
        )
    )
for shard_index, worker in workers:
    return_code = worker.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, ["generation-shard", str(shard_index)])

answered = completed_rows()
if QUESTION_LIMIT:
    print(f"partial run finished cleanly: {answered} answered under a {QUESTION_LIMIT} cap.")
    print("Save this version, then attach its output to the next session.")
elif answered < 1012:
    raise SystemExit(
        f"Session ended with {answered}/1012 answered. Nothing is lost: save this\n"
        "version, attach its output to a new session, and run this notebook again.\n"
        "Do not change the commit or the shard count."
    )


In [ ]:
if QUESTION_LIMIT:
    print("skipping merge and packaging: this session stopped at its cap")
else:
    subprocess.run(
        [
            sys.executable,
            "scripts/51_merge_generation_shards.py",
            *[str(path) for path in shard_dirs],
            "--output",
            str(GEN),
            "--expected-rows",
            "1012",
        ],
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            "scripts/45_finalize_submission.py",
            str(GEN / "submission.json"),
            "--retrieval",
            str(RETRIEVAL),
            "--manifest",
            str(MANIFEST.with_suffix(".parquet")),
            "--output",
            str(GEN / "submission_z4_abs.json"),
        ],
        check=True,
    )
    FINAL_SUBMISSION = GEN / "submission_z4_abs.json"
    subprocess.run(
        [
            sys.executable,
            "scripts/40_validate_submission.py",
            str(FINAL_SUBMISSION),
            "--questions",
            str(DATA_ROOT / "questions/questions.jsonl"),
            "--evidence-root",
            str(GEN),
            "--allow-partial-docs",
        ],
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            "scripts/41_package_submission.py",
            str(FINAL_SUBMISSION),
            "/kaggle/working/submission.zip",
            "--questions",
            str(DATA_ROOT / "questions/questions.jsonl"),
            "--evidence-root",
            str(GEN),
            "--allow-partial-docs",
        ],
        check=True,
    )
    predictions = json.loads(FINAL_SUBMISSION.read_text(encoding="utf-8"))
    traces = [
        json.loads(line)
        for line in (GEN / "program_traces.jsonl").read_text(encoding="utf-8").splitlines()
        if line
    ]
    fallbacks = [trace for trace in traces if trace.get("fallback")]
    assert len(predictions) == 1012
    submission = Path("/kaggle/working/submission.zip")
    print(
        f"predictions {len(predictions)}/1012, solved {len(predictions) - len(fallbacks)}, "
        f"fallback {len(fallbacks)}"
    )
    print(submission, submission.stat().st_size, sha256(submission))
